In [1]:
import os

In [2]:
%pwd

'/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject/research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject'

### Following the same process
- update config.yaml with data validation
- 

In [5]:
import pandas as pd

data = pd.read_csv(r"/workspaces/End-to-End-MLOps-Bootcamp-Build-Deploy-and-Automate-ML-with-Data-Science-Projects/myFirstNewProject/artifacts/data_ingestion/winequality-red.csv")

In [6]:
data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [7]:
data_info = pd.DataFrame(data.dtypes)

In [8]:
# Returns { 'column_name': 'dtype_string' }
dtype_dict = data.dtypes.apply(lambda x: x.name).to_dict()

In [9]:
dtype_dict

{'fixed acidity': 'float64',
 'volatile acidity': 'float64',
 'citric acid': 'float64',
 'residual sugar': 'float64',
 'chlorides': 'float64',
 'free sulfur dioxide': 'float64',
 'total sulfur dioxide': 'float64',
 'density': 'float64',
 'pH': 'float64',
 'sulphates': 'float64',
 'alcohol': 'float64',
 'quality': 'int64'}

In [10]:
data.isnull().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
dtype: int64

In [11]:
data.shape

(1599, 12)

In [12]:
from dataclasses import dataclass
from pathlib import Path

## Step-1: updated config.yaml
## Step-2: update params.yaml
## Step-3: updated schema.yaml
## Step-4: updating entity module for class creation that can configured in configuration manager.
@dataclass
class DataValidationConfig:
    root_dir: Path
    unzip_data_dir: Path
    STATUS_FILE: Path
    all_schema: dict

## Step-5: above; updated with module of configuration, now Updating configuration manager
from src.my_first_end_to_end_project.constants import *
from src.my_first_end_to_end_project.utils.common_utils import read_yaml,create_directories

class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH,
            schema_filepath = SCHEMA_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        create_directories([self.config.artifacts_root])
    
    def get_data_validation_config(self)-> DataValidationConfig:
        config = self.config.data_validation
        schema = self.schema.COLUMNS

        create_directories([config.root_dir])

        data_validation_config = DataValidationConfig(
            root_dir = config.root_dir,
            STATUS_FILE=config.STATUS_FILE,
            unzip_data_dir= config.unzip_data_dir,
            all_schema=schema,
        )

        return data_validation_config

## Step-6: Now creating the data validation class post configuring
import os
from src.my_first_end_to_end_project.logger import logger

class DataValidation:
    def __init__(self, config:DataValidationConfig):
        self.config = config

    # def validate_all_columns(self)-> bool:
    #     try:
    #         validation_status = None

    #         data = pd.read_csv(self.config.unzip_data_dir)
    #         all_columns = list(data.columns)

            
    #         all_schema = self.config.all_schema.keys()

    #         for col in all_columns:
    #             if col not in all_schema:
    #                 validation_status = False
    #                 with open(self.config.STATUS_FILE, 'a') as f:
    #                     f.write(f"for {col} --> Validation status: {validation_status} \n")

    #             else:
    #                 validation_status= True
    #                 with open(self.config.STATUS_FILE,'a') as f:
    #                     f.write(f"for {col} --> Validation status: {validation_status} \n")

    #         return validation_status

    #     except Exception as e:
    #         raise e

    def validate_all_columns(self)-> bool:
        try:
            validation_status = True
            data = pd.read_csv(self.config.unzip_data_dir)
            all_cols = list(data.columns)

            schema = self.config.all_schema

            with open(self.config.STATUS_FILE, 'w') as f:
                # check all cols present in csv are in schema
                for col in all_cols:
                    if col not in schema:
                        validation_status = False
                        f.write(f"Column: {col} | Validation status: {validation_status} \n")
                    else:
                        expected_col_type = schema[col]
                        actual_col_type = str(data[col].dtype)

                        if expected_col_type != actual_col_type:
                            validation_status = False
                            f.write(f"Column: {col} | Validation status: {validation_status} | Data type mismatch (expected type {expected_col_type}, but actual data type {actual_col_type})\n")
                        else:
                            validation_status = True
                            f.write(f"Column: {col} | Validation status: {validation_status} | Data type MATCH (expected type {expected_col_type} == actual data type {actual_col_type}) \n")

                # check all cols present in schema present in csv or not
                for col in schema.keys():
                    if col not in all_cols:
                        validation_status = False
                        f.write(f"Column: {col} | Column missing in .csv file | Validation status : {validation_status} \n") 
            return validation_status
        except Exception as e:
            raise e

In [13]:
## Step-7: testing the class return
try:
    config = ConfigurationManager()
    data_validation_config = config.get_data_validation_config()
    data_validation = DataValidation(data_validation_config)
    data_validation.validate_all_columns()
except Exception as e:
    raise e

2026-01-05 13:47:46 - INFO - common_utils - yaml file: config/config.yaml is loaded successfully 🥳
2026-01-05 13:47:46 - INFO - common_utils - yaml file: params.yaml is loaded successfully 🥳
2026-01-05 13:47:46 - INFO - common_utils - yaml file: schema.yaml is loaded successfully 🥳
2026-01-05 13:47:46 - INFO - common_utils - Created directory Successfuly at: artifacts 🥳
2026-01-05 13:47:46 - INFO - common_utils - Created directory Successfuly at: artifacts/data_validation 🥳


In [ ]:
## Step-8: Converting them into modular coding
### Step-8.1: Move to entity and create dataclass there
### Step-8.2: Update configuration manager
### Step-8.3: Move to component and create component called data_validation.py
### Step-8.4: Create pipeline "data_validation"
### Step-8.5: Update main.py

